# FE model

In [2]:
# import os
# import yaml
# import torch
# from credit.models import load_model
# from credit.parser import credit_main_parser

In [3]:
# CONFIG_FILE_DIR = '/glade/u/home/ksha/miles-physics/config/'

## Model dev section

In [4]:
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
import logging
logger = logging.getLogger(__name__)

In [6]:
from timm.models.swin_transformer_v2 import SwinTransformerV2Stage

In [18]:
import math

In [24]:
def apply_spectral_norm(model):
    """
    add spectral norm to all the conv and linear layers
    """
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear, nn.ConvTranspose2d)):
            nn.utils.spectral_norm(module)

def round_up_to_patch_multiple(size, patch_size):
    """Round each dimension up to the next multiple of patch_size."""
    return [(s + patch_size - 1) // patch_size * patch_size for s in size]

class PatchEmbed3D(nn.Module):
    def __init__(self,
                 patch_size      = 16,
                 temporal_patch  = 1,
                 in_chans        = 3,
                 embed_dim       = 768):
        super().__init__()

        # allow patch_size to be int or (ph, pw)
        if isinstance(patch_size, int):
            patch_size = (patch_size, patch_size)

        self.patch_size      = patch_size      # (ph, pw)
        self.temporal_patch  = temporal_patch  # pt

        self.proj = nn.Conv3d(
            in_chans,
            embed_dim,
            kernel_size=(temporal_patch, *patch_size),
            stride     =(temporal_patch, *patch_size)
        )

    def forward(self, x: torch.Tensor):
        # x: (B, C, T, H, W)
        x = self.proj(x)                 # (B, embed_dim, T', Hp, Wp)
        B, C, Tp, Hp, Wp = x.shape

        # flatten spatial dimensions, keep Tp separate for now
        x = x.permute(0, 2, 3, 4, 1)     # (B, Tp, Hp, Wp, C)
        x = x.reshape(B * Tp, Hp * Wp, C)  # (B*Tp, N, C)

        return x, (Hp, Wp, Tp)

class RelativePositionBias2D(nn.Module):
    def __init__(self, num_heads, max_hw=256):
        super().__init__()
        self.num_heads = num_heads
        self.max_hw = max_hw

        # Maximum possible number of relative positions
        self.relative_bias_table = nn.Parameter(
            torch.zeros((2 * max_hw - 1) * (2 * max_hw - 1), num_heads)
        )

        self.max_pos = max_hw

    def get_bias_index(self, Hp, Wp, device):
        coords_h = torch.arange(Hp, device=device)
        coords_w = torch.arange(Wp, device=device)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'))  # (2, H, W)
        coords_flat = coords.flatten(1)
        rel_coords = coords_flat[:, :, None] - coords_flat[:, None, :]  # (2, N, N)
        rel_coords = rel_coords.permute(1, 2, 0).contiguous()

        rel_coords[:, :, 0] += self.max_pos - 1
        rel_coords[:, :, 1] += self.max_pos - 1
        rel_coords[:, :, 0] *= 2 * self.max_pos - 1

        return rel_coords.sum(-1)  # (N, N)

    def forward(self, Hp, Wp):
        N = Hp * Wp
        index = self.get_bias_index(Hp, Wp, device=self.relative_bias_table.device)
        bias = self.relative_bias_table[index.view(-1)].view(N, N, -1)
        return bias.permute(2, 0, 1)  # (num_heads, N, N)

class AttentionWithRPE(nn.Module):
    def __init__(self, dim, num_heads, window_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.rpb = RelativePositionBias2D(num_heads=num_heads)

    def forward(self, x, Hp, Wp):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv  # each: (B, heads, N, head_dim)
    
        attn = (q @ k.transpose(-2, -1)) * self.scale
        rpb = self.rpb(Hp, Wp)  # shape: (heads, N, N)
        attn = attn + rpb.unsqueeze(0)  # (1, heads, N, N)
        attn = attn.softmax(dim=-1)
    
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = AttentionWithRPE(dim, num_heads, window_size)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim)
        )

    def forward(self, x, Hp, Wp):
        x = x + self.attn(self.norm1(x), Hp, Wp)
        x = x + self.mlp(self.norm2(x))
        return x

class ViTWithRPE(nn.Module):
    def __init__(
        self, 
        patch_size=16,
        frame=2,
        in_chans=90, 
        out_chans=100, 
        embed_dim=1024, 
        depth=24, 
        num_heads=8,
        use_spectral_norm=True,
        max_size=(512, 512),
        padding_conf=None,
        post_conf=None,
        **kwargs,
    ):
        super().__init__()
        
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.out_chans = out_chans
        self.embed_dim = embed_dim
        self.use_spectral_norm = use_spectral_norm
        #self.patch_embed = PatchEmbed(patch_size, in_chans, embed_dim)
        self.patch_embed = PatchEmbed3D(patch_size, frame, in_chans, embed_dim)
        
        grid_size = (max_size[0] // patch_size, max_size[1] // patch_size)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, window_size=None)
            for _ in range(depth)
        ])

        self.output_proj = nn.Linear(embed_dim, patch_size * patch_size * out_chans)

        if self.use_spectral_norm:
            logger.info("Adding spectral norm to all conv and linear layers")
            apply_spectral_norm(self)

        if post_conf is None:
            post_conf = {"activate": False}

        self.use_post_block = post_conf["activate"]
        
        if self.use_post_block:
            self.postblock = PostBlock(post_conf)
    
    def forward(self, x):
        
        x_copy = None
        
        if self.use_post_block:
            x_copy = x.clone().detach()
        
        B, C, T, H, W = x.shape
        new_H, new_W  = round_up_to_patch_multiple((H, W), self.patch_size)

        # resize spatially if necessary, frame by frame
        if (H != new_H) or (W != new_W):
            x = x.reshape(B * T, C, H, W)               # (B*T, C, H, W)
            x = F.interpolate(x, size=(new_H, new_W), mode='bilinear', align_corners=False)
            x = x.reshape(B, C, T, new_H, new_W)
            #H, W = new_H, new_W                         # keep invariants

        # ── Patch embed (3‑D) ────────────────────────────────────────────
        x_tokens, (Hp, Wp, Tp) = self.patch_embed(x)     # Tp might be < T if temporal_patch>1
        # x_tokens : (B*Tp, Hp*Wp, embed_dim)
        print(x_tokens.shape)
        # ── Transformer (unchanged) ─────────────────────────────────────
        for blk in self.blocks:
            x_tokens = blk(x_tokens, Hp, Wp)

        x_tokens = self.output_proj(x_tokens)            # (B*Tp, N, P²*out_chans)
        
        # ── Restore to images frame by frame ────────────────────────────
        x_tokens = x_tokens.view(B * Tp, Hp, Wp, self.out_chans, self.patch_size, self.patch_size)
        x_tokens = x_tokens.permute(0, 3, 1, 4, 2, 5).contiguous()
        x_tokens = x_tokens.view(B * Tp, self.out_chans, new_H, new_W)

        # undo time‑into‑batch folding
        x_out = x_tokens.view(B, self.out_chans, new_H, new_W)
        #x_out = x_out.permute(0, 2, 1, 3, 4)             # (B, out_chans, Tp, H, W)

        # Optional: downsample back to original size
        if (H != new_H) or (W != new_W):
            x_out = F.interpolate(x_out, size=(H, W), mode='bilinear', align_corners=False)

        x_out = x_out.unsqueeze(2)
        
        if self.use_post_block:
            x_out = {
                "y_pred": x_out,
                "x": x_copy,
            }
            x_out = self.postblock(x_out)
            
        return x_out

In [25]:
# def apply_spectral_norm(model):
#     """
#     add spectral norm to all the conv and linear layers
#     """
#     for module in model.modules():
#         if isinstance(module, (nn.Conv2d, nn.Linear, nn.ConvTranspose2d)):
#             nn.utils.spectral_norm(module)

# def round_up_to_patch_multiple(size, patch_size):
#     """Round each dimension up to the next multiple of patch_size."""
#     return [(s + patch_size - 1) // patch_size * patch_size for s in size]

# class PatchEmbed3D(nn.Module):
#     def __init__(self,
#                  patch_size      = 16,
#                  temporal_patch  = 1,
#                  in_chans        = 3,
#                  embed_dim       = 768):
#         super().__init__()

#         # allow patch_size to be int or (ph, pw)
#         if isinstance(patch_size, int):
#             patch_size = (patch_size, patch_size)

#         self.patch_size      = patch_size      # (ph, pw)
#         self.temporal_patch  = temporal_patch  # pt

#         self.proj = nn.Conv3d(
#             in_chans,
#             embed_dim,
#             kernel_size=(temporal_patch, *patch_size),
#             stride     =(temporal_patch, *patch_size)
#         )

#     def forward(self, x: torch.Tensor):
#         # x: (B, C, T, H, W)
#         x = self.proj(x)                 # (B, embed_dim, T', Hp, Wp)
#         B, C, Tp, Hp, Wp = x.shape

#         # flatten spatial dimensions, keep Tp separate for now
#         x = x.permute(0, 2, 3, 4, 1)     # (B, Tp, Hp, Wp, C)
#         x = x.reshape(B * Tp, Hp * Wp, C)  # (B*Tp, N, C)

#         return x, (Hp, Wp, Tp)

# class RelativePositionBias2D(nn.Module):
#     def __init__(self, num_heads, max_hw=256):
#         super().__init__()
#         self.num_heads = num_heads
#         self.max_hw = max_hw

#         # Maximum possible number of relative positions
#         self.relative_bias_table = nn.Parameter(
#             torch.zeros((2 * max_hw - 1) * (2 * max_hw - 1), num_heads)
#         )

#         self.max_pos = max_hw

#     def get_bias_index(self, Hp, Wp, device):
#         coords_h = torch.arange(Hp, device=device)
#         coords_w = torch.arange(Wp, device=device)
#         coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'))  # (2, H, W)
#         coords_flat = coords.flatten(1)
#         rel_coords = coords_flat[:, :, None] - coords_flat[:, None, :]  # (2, N, N)
#         rel_coords = rel_coords.permute(1, 2, 0).contiguous()

#         rel_coords[:, :, 0] += self.max_pos - 1
#         rel_coords[:, :, 1] += self.max_pos - 1
#         rel_coords[:, :, 0] *= 2 * self.max_pos - 1

#         return rel_coords.sum(-1)  # (N, N)

#     def forward(self, Hp, Wp):
#         N = Hp * Wp
#         index = self.get_bias_index(Hp, Wp, device=self.relative_bias_table.device)
#         bias = self.relative_bias_table[index.view(-1)].view(N, N, -1)
#         return bias.permute(2, 0, 1)  # (num_heads, N, N)

# class AttentionWithRPE(nn.Module):
#     def __init__(self, dim, num_heads, window_size):
#         super().__init__()
#         self.num_heads = num_heads
#         self.head_dim = dim // num_heads
#         self.scale = self.head_dim ** -0.5
#         self.qkv = nn.Linear(dim, dim * 3)
#         self.proj = nn.Linear(dim, dim)
#         self.rpb = RelativePositionBias2D(num_heads=num_heads)

#     def forward(self, x, Hp, Wp):
#         B, N, C = x.shape
#         qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
#         q, k, v = qkv  # each: (B, heads, N, head_dim)
    
#         attn = (q @ k.transpose(-2, -1)) * self.scale
#         rpb = self.rpb(Hp, Wp)  # shape: (heads, N, N)
#         attn = attn + rpb.unsqueeze(0)  # (1, heads, N, N)
#         attn = attn.softmax(dim=-1)
    
#         x = (attn @ v).transpose(1, 2).reshape(B, N, C)
#         return self.proj(x)

# class SwinV2Block(nn.Module):
#     """
#     A 'single‑layer stage' that mimics the old TransformerBlock interface:
#         forward(x, Hp, Wp) -> x      (shapes unchanged)
#     """

#     def __init__(
#         self,
#         dim: int,
#         num_heads: int,
#         window_size: int = 7,   # default – will adapt at runtime
#         mlp_ratio: float = 4.0,
#         drop_path: float = 0.0,
#     ):
#         super().__init__()
#         self.init_window_size = window_size   # remember original wish

#         # dummy resolution; will be overwritten the first time forward() runs
#         self.stage = SwinTransformerV2Stage(
#             dim              = dim,
#             out_dim          = dim,          # no channel change (no downsample)
#             input_resolution = (1, 1),       # placeholder
#             depth            = 1,            # exactly one block
#             num_heads        = num_heads,
#             window_size      = window_size,
#             mlp_ratio        = mlp_ratio,
#             qkv_bias         = True,
#             proj_drop        = 0.0,
#             attn_drop        = 0.0,
#             drop_path        = drop_path,
#             norm_layer       = nn.LayerNorm,
#             downsample       = None,
#         )

#     # ------------------------------------------------------------------ #
#     # helpers
#     # ------------------------------------------------------------------ #
#     @staticmethod
#     def _gcd_window(Hp: int, Wp: int, max_ws: int) -> int:
#         """
#         Return the largest window ≤ `max_ws` that divides both Hp & Wp.
#         If none exists, return the GCD (may be 1).
#         """
#         g = math.gcd(Hp, Wp)
#         for k in range(min(max_ws, g), 1, -1):
#             if g % k == 0:
#                 return k
#         return g  # 1 if Hp & Wp are co‑prime

#     def _reconfigure_blocks(self, Hp: int, Wp: int, ws: int):
#         """
#         Push the current (Hp, Wp) and window/shift sizes into the single
#         Swin block contained in the stage.
#         """
#         blk = self.stage.blocks[0]                 # depth == 1
#         blk.input_resolution = (Hp, Wp)
#         blk.window_size      = ws
#         blk.shift_size       = 0                  # depth==1 ⇒ no shift
#         # also fix attention sub‑module
#         blk.attn.window_size = ws

#     # ------------------------------------------------------------------ #
#     # forward
#     # ------------------------------------------------------------------ #
#     def forward(self, x: torch.Tensor, Hp: int, Wp: int) -> torch.Tensor:
#         # 1) pick a legal window
#         ws = self._gcd_window(Hp, Wp, self.init_window_size)

#         # 2) adapt stored metadata (resolution + window/shift sizes)
#         if (Hp, Wp) != self.stage.input_resolution or ws != self.stage.blocks[0].window_size:
#             self.stage.input_resolution = (Hp, Wp)
#             self._reconfigure_blocks(Hp, Wp, ws)

#         # 3) run the single‑layer stage; timm returns (x, (Hp, Wp))
#         x, _ = self.stage(x, (Hp, Wp))
#         return x

# class ViTWithRPE(nn.Module):
#     def __init__(
#         self, 
#         patch_size=16,
#         frame=2,
#         in_chans=90, 
#         out_chans=100, 
#         embed_dim=1024, 
#         depth=24, 
#         num_heads=8,
#         use_spectral_norm=True,
#         max_size=(512, 512),
#         padding_conf=None,
#         post_conf=None,
#         **kwargs,
#     ):
#         super().__init__()
        
#         self.patch_size = patch_size
#         self.in_chans = in_chans
#         self.out_chans = out_chans
#         self.embed_dim = embed_dim
#         self.use_spectral_norm = use_spectral_norm
#         #self.patch_embed = PatchEmbed(patch_size, in_chans, embed_dim)
#         self.patch_embed = PatchEmbed3D(patch_size, frame, in_chans, embed_dim)
        
#         grid_size = (max_size[0] // patch_size, max_size[1] // patch_size)

#         self.blocks = nn.ModuleList([
#             SwinV2Block(
#                 dim        = embed_dim,
#                 num_heads  = num_heads,
#                 window_size=7,      # any preferred starting value
#                 drop_path  = 0.0,
#             )
#             for _ in range(depth)
#         ])
                
#         self.output_proj = nn.Linear(embed_dim, patch_size * patch_size * out_chans)

#         if self.use_spectral_norm:
#             logger.info("Adding spectral norm to all conv and linear layers")
#             apply_spectral_norm(self)

#         if post_conf is None:
#             post_conf = {"activate": False}

#         self.use_post_block = post_conf["activate"]
        
#         if self.use_post_block:
#             self.postblock = PostBlock(post_conf)
    
#     def forward(self, x):
        
#         x_copy = None
        
#         if self.use_post_block:
#             x_copy = x.clone().detach()
        
#         B, C, T, H, W = x.shape
#         new_H, new_W  = round_up_to_patch_multiple((H, W), self.patch_size)

#         # resize spatially if necessary, frame by frame
#         if (H != new_H) or (W != new_W):
#             x = x.reshape(B * T, C, H, W)               # (B*T, C, H, W)
#             x = F.interpolate(x, size=(new_H, new_W), mode='bilinear', align_corners=False)
#             x = x.reshape(B, C, T, new_H, new_W)
#             #H, W = new_H, new_W                         # keep invariants

#         # ── Patch embed (3‑D) ────────────────────────────────────────────
#         x_tokens, (Hp, Wp, Tp) = self.patch_embed(x)     # Tp might be < T if temporal_patch>1
#         # x_tokens : (B*Tp, Hp*Wp, embed_dim)

#         print(x_tokens.shape)
#         raise
#         # ── Transformer (unchanged) ─────────────────────────────────────
#         for blk in self.blocks:
#             x_tokens = blk(x_tokens, Hp, Wp)

#         x_tokens = self.output_proj(x_tokens)            # (B*Tp, N, P²*out_chans)
        
#         # ── Restore to images frame by frame ────────────────────────────
#         x_tokens = x_tokens.view(B * Tp, Hp, Wp, self.out_chans, self.patch_size, self.patch_size)
#         x_tokens = x_tokens.permute(0, 3, 1, 4, 2, 5).contiguous()
#         x_tokens = x_tokens.view(B * Tp, self.out_chans, new_H, new_W)

#         # undo time‑into‑batch folding
#         x_out = x_tokens.view(B, self.out_chans, new_H, new_W)
#         #x_out = x_out.permute(0, 2, 1, 3, 4)             # (B, out_chans, Tp, H, W)

#         # Optional: downsample back to original size
#         if (H != new_H) or (W != new_W):
#             x_out = F.interpolate(x_out, size=(H, W), mode='bilinear', align_corners=False)

#         x_out = x_out.unsqueeze(2)
        
#         if self.use_post_block:
#             x_out = {
#                 "y_pred": x_out,
#                 "x": x_copy,
#             }
#             x_out = self.postblock(x_out)
            
#         return x_out

In [26]:
torch.Size([1, 120, 1024])

torch.Size([1, 120, 1024])

In [28]:
model = ViTWithRPE(patch_size=16)

for shape in [(123, 77),]:
    x = torch.randn(1, 90, 2, *shape)
    y = model(x)
    print(f"Input: {x.shape} → Output: {y.shape}")

torch.Size([1, 40, 1024])
Input: torch.Size([1, 90, 2, 123, 77]) → Output: torch.Size([1, 100, 1, 123, 77])


In [21]:
2*90*123*234

5180760

In [18]:
model = ViTWithRPE(patch_size=4, max_size=(1008, 1008))
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {n_params:,}")

Total trainable parameters: 355,560,192


## FuXi param check

In [21]:
# old rollout config
#config_name = '/glade/u/home/ksha/miles-credit/config/example_physics_single.yml'

config_name = '/glade/work/ksha/DWC_runs/CONUS_GP_base/model_single.yml'

# Read YAML file
with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [22]:
conf = credit_main_parser(conf, parse_training=True, parse_predict=False, print_summary=True)

Upper-air variables: ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q']
Surface variables: ['WRF_SP', 'WRF_T2', 'WRF_TD2', 'WRF_U10', 'WRF_V10']
Dynamic forcing variables: []
Diagnostic variables: []
Forcing variables: []
Static variables: ['z_norm', 'var_norm', 'lu_norm', 'LANDMASK']


In [23]:
# conf['model']['post_conf']['activate'] = False
# conf['model']['interp'] = False
# conf['model']['padding_conf']['pad_lat'] = [21, 22]
# conf['model']['padding_conf']['pad_lon'] = [44, 44]
# conf['model']['param_interior']['image_height'] = 280
# conf['model']['param_interior']['image_width'] = 392
# conf['model']['padding_conf']['activate'] = False

In [24]:
conf['model']['post_conf']['activate'] = False

image_height_inside = conf['model']['param_interior']['image_height']
image_width_inside = conf['model']['param_interior']['image_width']
levels_inside = conf['model']['param_interior']['levels']
frames_inside = conf['model']['param_interior']['frames']
channels_inside = conf['model']['param_interior']['channels']
surface_channels_inside = conf['model']['param_interior']['surface_channels']
input_only_channels_inside = conf['model']['param_interior']['input_only_channels']
output_only_channels_inside = conf['model']['param_interior']['output_only_channels']

image_height_outside = conf['model']['param_outside']['image_height']
image_width_outside = conf['model']['param_outside']['image_width']
levels_outside = conf['model']['param_outside']['levels']
frames_outside = conf['model']['param_outside']['frames']
channels_outside = conf['model']['param_outside']['channels']
surface_channels_outside = conf['model']['param_outside']['surface_channels']


# ============================================================= #
# build the model
model = WRF_Tansformer(**conf['model']).to("cuda")

# ============================================================= #
# test the model

# pass an input tensor to test the graph
input_tensor_inside = torch.randn(
    1, channels_inside * levels_inside + surface_channels_inside + input_only_channels_inside, 
    frames_inside, 
    image_height_inside, 
    image_width_inside
).to("cuda")    

input_tensor_outside = torch.randn(
    1, channels_outside * levels_outside + surface_channels_outside, 
    frames_outside, 
    image_height_outside, 
    image_width_outside
).to("cuda") 

input_time = torch.randn(1, 16).to("cuda")  

print('Input shape inside: {}'.format(input_tensor_inside.shape))
print('Input shape outside: {}'.format(input_tensor_outside.shape))
print('Input shape time: {}'.format(input_time.shape))

y_pred = model(input_tensor_inside, input_tensor_outside, input_time)
print("Predicted shape: {}".format(y_pred.shape))

Input shape inside: torch.Size([1, 89, 1, 336, 336])
Input shape outside: torch.Size([1, 85, 2, 336, 336])
Input shape time: torch.Size([1, 16])
(0, 0, 0, 0)


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 15.89 GiB of which 31.25 MiB is free. Including non-PyTorch memory, this process has 15.78 GiB memory in use. Of the allocated memory 15.40 GiB is allocated by PyTorch, and 78.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## FuXi I/O size and padding

**Auto-detect pad size**

In [1]:
import math

In [2]:
def compute_padding_sizes(image_height, image_width, window_size, patch_height, patch_width, base_val=40, N_lat_add=0, N_lon_add=0):
    """
    Computes the required padding sizes pad_lat and pad_lon for given image dimensions,
    window size, and patch sizes, such that after padding:
    - The input resolutions used in the model are divisible by window_size.
    - After unpadding and any interpolation, the output tensor size is the same as the original.
    - The padding sizes are as close as possible to [base_val, base_val].
    - Allows adjustment of required input resolutions by adding multiples of window_size.
    
    Args:
        image_height (int): Original image height.
        image_width (int): Original image width.
        window_size (int): Window size used in the model.
        patch_height (int): Patch height.
        patch_width (int): Patch width.
        base_val (int): Desired base padding value for each side.
        N_lat_add (int): Additional window sizes to add to required input resolution for latitude.
        N_lon_add (int): Additional window sizes to add to required input resolution for longitude.
    
    Returns:
        tuple: pad_lat (list[int]), pad_lon (list[int])
               Padding sizes for latitude and longitude ([top, bottom], [left, right]).
    """
    frames = 2
    frame_patch_size = 2

    # Calculate initial input resolutions without padding
    input_resolution_lat = image_height / patch_height / 2
    input_resolution_lon = image_width / patch_width / 2

    # Calculate minimal required input resolutions that are divisible by window_size
    N_lat_min = math.ceil(input_resolution_lat / window_size)
    N_lon_min = math.ceil(input_resolution_lon / window_size)

    # Adjust required input resolutions by adding additional window sizes
    N_lat = N_lat_min + N_lat_add
    N_lon = N_lon_min + N_lon_add

    required_input_resolution_lat = N_lat * window_size
    required_input_resolution_lon = N_lon * window_size

    # Adjusted image dimensions after padding
    image_height_padded = required_input_resolution_lat * patch_height * 2
    image_width_padded = required_input_resolution_lon * patch_width * 2

    # Calculate total padding required
    pad_lat_total = int(image_height_padded - image_height)
    pad_lon_total = int(image_width_padded - image_width)

    # Check if total padding is non-negative
    if pad_lat_total < 0 or pad_lon_total < 0:
        return None, None

    # Distribute padding for latitude (height)
    pad_lat = distribute_padding(pad_lat_total, base_val)

    # Distribute padding for longitude (width)
    pad_lon = distribute_padding(pad_lon_total, base_val)

    return pad_lat, pad_lon

def distribute_padding(total_padding, base_val):
    if total_padding == 0:
        return [0, 0]
    
    # Distribute padding evenly
    pad_first = total_padding // 2
    pad_second = total_padding - pad_first

    # Adjust padding to be as close as possible to base_val
    if pad_first > base_val:
        pad_first = base_val
        pad_second = total_padding - pad_first
    if pad_second > base_val:
        pad_second = base_val
        pad_first = total_padding - pad_second

    # Ensure total padding matches
    if pad_first + pad_second != total_padding:
        pad_second += total_padding - (pad_first + pad_second)

    return [pad_first, pad_second]

In [12]:
compute_padding_sizes(280, 560, 7, 4, 4, base_val=0, N_lat_add=2, N_lon_add=2)

([112, 0], [112, 0])

In [9]:
56*5

280